# SciCap GRPO — Cycle Reconstruction Eval

**What this notebook does (the CycleCap "reconstruction" check):**

```
original figure  ──►  GRPO Qwen2.5-VL  ──►  caption  ──►  text-to-image  ──►  reconstructed figure
        │                                                                              │
        └──────────────────────  CLIP image–image cosine similarity  ◄────────────────┘
```

The intuition: if the generated caption captured the *distinguishing* content of the
figure, a text-to-image model fed only that caption should regenerate something a
vision encoder still considers close to the original. Higher cosine = more faithful caption.

**Dataset is loaded exactly as in the training notebook** (same Kaggle dataset,
same CSV, same path/basename logic). Validation = held-out rows the model never
trained on (SFT used 0–2000, GRPO used 0–500), so we evaluate on rows ≥ 2000.

> Note: general text-to-image models reconstruct *scientific* figures only loosely —
> treat the CLIP score as a **relative** signal (compare models / captions), not an
> absolute "is it the same plot" number. CLIPScore(caption, original) is reported too
> as a direct grounding metric.

In [ ]:
# ── Install deps (Unsloth for the VLM, diffusers for reconstruction) ───────
import os
if not os.path.exists("/content/.recon_deps"):
    !pip install --quiet --upgrade pip
    !pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --quiet "bitsandbytes>=0.46.0" "diffusers>=0.30.0" "accelerate" "transformers"
    !pip install --quiet evaluate scikit-image matplotlib
    open("/content/.recon_deps", "w").close()
    print("✅ Installed. If Unsloth asks, Runtime > Restart session, then re-run from here.")
else:
    print("✅ Deps already installed.")

In [ ]:
# ── Kaggle credentials + dataset download (identical to training notebook) ─
from google.colab import files
import os

if not os.path.exists("kaggle.json"):
    print("Select your kaggle.json to upload.")
    files.upload()

os.environ["KAGGLE_CONFIG_DIR"] = os.getcwd()
if not os.path.exists("scicap_data"):
    !kaggle datasets download -d dhivyaraman123/scicap-dataset
    !unzip -q scicap-dataset.zip -d scicap_data
print("✅ Dataset ready.")

In [ ]:
# ── Load SciCap exactly as in the training notebook, take HELD-OUT split ───
import os
import pandas as pd
from datasets import Dataset, Image

csv_path  = "/content/scicap_data/scicap_train.csv"
image_dir = "/content/scicap_data/scicap_images_compressed/share-task-img-mask/arxiv/train"

df = pd.read_csv(csv_path)
df["image"] = df["image"].astype(str).str.strip().map(os.path.basename)
existing = set(os.listdir(image_dir))
df = df[df["image"].isin(existing)].copy()
df["image_path"] = image_dir + os.sep + df["image"]
print(f"Found {len(df)} valid image-caption pairs")

# Validation = rows the model never saw. SFT trained 0-2000, GRPO trained 0-500.
VAL_START = 2000          # first held-out row
N_EVAL    = 50            # how many to evaluate (raise for a fuller run)

val_df = df.iloc[VAL_START:VAL_START + N_EVAL].reset_index(drop=True)
val_ds = Dataset.from_dict({
    "image":   val_df["image_path"].tolist(),
    "caption": val_df["caption"].astype(str).tolist(),
}).cast_column("image", Image())
print(val_ds)
assert len(val_ds) > 0, "Empty val set — check VAL_START vs dataset size."

# --- OPTIONAL: if your Kaggle copy ships a real validation CSV, point to it here ---
# val_csv = "/content/scicap_data/scicap_val.csv"
# if os.path.exists(val_csv): ... (same basename/existing/cast logic as above)

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
from unsloth import FastVisionModel
import torch

GRPO_MODEL = "shemalfoy/qwen2-vl-scicap-grpo"   # or "...-grpo-v2"

model, tokenizer = FastVisionModel.from_pretrained(
    GRPO_MODEL,
    max_seq_length = 16384,
    load_in_4bit = True,
    fast_inference = False,
)

# Same pixel budget as training so captions use the resolution the model was
# tuned for. NOTE: set via image_processor.size{} -- min_pixels/max_pixels are
# read-only properties in current transformers.
min_pixels, max_pixels = 256 * 28 * 28, 1280 * 28 * 28
def set_pixel_budget(proc, mn, mx):
    # In current transformers, Qwen2VLImageProcessor.min_pixels/max_pixels are
    # READ-ONLY properties; the real source of truth is image_processor.size
    # = {"shortest_edge": min_pixels, "longest_edge": max_pixels}. smart_resize
    # reads those at preprocess time. We set size first, then try the legacy
    # attributes (harmlessly skipped on versions where they're read-only).
    ip = getattr(proc, "image_processor", proc)
    try:
        ip.size = {"shortest_edge": mn, "longest_edge": mx}
    except Exception as e:
        print("size set failed:", e)
    for obj in (ip, proc):
        for attr, val in (("min_pixels", mn), ("max_pixels", mx)):
            try:
                setattr(obj, attr, val)
            except (AttributeError, TypeError):
                pass   # read-only property -> size{} already handles it
    return ip

_ip = set_pixel_budget(tokenizer, min_pixels, max_pixels)
FastVisionModel.for_inference(model)
print(f"✅ {GRPO_MODEL} loaded | size={_ip.size}")

In [ ]:
# ── Caption generation helper (same prompt used in training) ───────────────
def generate_caption(image, max_new_tokens=128):
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Describe this scientific figure."},
    ]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(image.convert("RGB"), text,
                       add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# quick smoke test on one sample
_s = val_ds[0]
print("CAPTION[0]:", generate_caption(_s["image"])[:200])

In [ ]:
# ── PHASE 1: caption every validation image (VLM in memory) ────────────────
from PIL import Image as PILImage

originals, captions, refs = [], [], []
for i in range(len(val_ds)):
    ex = val_ds[i]
    img = ex["image"].convert("RGB")
    originals.append(img)
    captions.append(generate_caption(img))
    refs.append(ex["caption"])
    if i % 10 == 0:
        print(f"[caption] {i}/{len(val_ds)}  {captions[-1][:70]!r}")
print(f"✅ Generated {len(captions)} captions.")

# Free the 7B VLM before loading the diffusion model (helps on 16GB GPUs).
# Comment out the next 3 lines if you have plenty of VRAM (A100/L4).
import gc, torch
del model; gc.collect(); torch.cuda.empty_cache()
print("Freed VLM from VRAM.")

In [ ]:
# ── PHASE 2: reconstruct each image from its caption (text-to-image) ───────
import torch
from diffusers import AutoPipelineForText2Image

# sd-turbo: fast (1-4 steps), fits a T4. Swap for SDXL-Turbo on bigger GPUs.
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

def reconstruct(caption, steps=2):
    prompt = caption.strip() or "a scientific figure"
    return pipe(prompt=prompt, num_inference_steps=steps,
                guidance_scale=0.0, height=512, width=512).images[0]

recons = []
for i, cap in enumerate(captions):
    recons.append(reconstruct(cap))
    if i % 10 == 0:
        print(f"[reconstruct] {i}/{len(captions)}")
print(f"✅ Reconstructed {len(recons)} images.")

In [ ]:
# ── Score: CLIP image–image (cycle fidelity) + CLIPScore(caption, original)─
import torch, numpy as np
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval()
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def _as_tensor(out, kind="image"):
    # get_image_features normally returns a Tensor, but Unsloth (imported above)
    # patches transformers and some builds return a ModelOutput instead. Unwrap
    # to the embedding tensor. Both orig & recon go through the same path, so the
    # representation choice is consistent for the cosine comparison.
    if torch.is_tensor(out):
        return out
    for attr in (f"{kind}_embeds", "pooler_output", "last_hidden_state"):
        v = getattr(out, attr, None)
        if v is not None:
            return v.mean(dim=1) if attr == "last_hidden_state" else v
    return out[0]

@torch.no_grad()
def img_embed(img):
    inp = clip_proc(images=[img.convert("RGB")], return_tensors="pt").to("cuda")
    e = _as_tensor(clip_model.get_image_features(**inp), "image")
    return torch.nn.functional.normalize(e, dim=-1)

@torch.no_grad()
def clipscore(caption, img):
    if not caption.strip():
        return 0.0
    inp = clip_proc(text=[caption], images=[img.convert("RGB")], return_tensors="pt",
                    padding=True, truncation=True, max_length=77).to("cuda")
    out = clip_model(**inp)
    lpi = out.logits_per_image if hasattr(out, "logits_per_image") else out[0]
    return lpi.squeeze().item() / 100.0

cycle_sims, ground_scores = [], []
for orig, recon, cap in zip(originals, recons, captions):
    sim = (img_embed(orig) * img_embed(recon)).sum().item()   # cosine (unit vecs)
    cycle_sims.append(sim)
    ground_scores.append(clipscore(cap, orig))

cycle_sims, ground_scores = np.array(cycle_sims), np.array(ground_scores)
print("=== Cycle reconstruction over {} held-out figures ===".format(len(originals)))
print(f"Mean CLIP image–image cosine (orig vs reconstructed): {cycle_sims.mean():.4f}")
print(f"Mean CLIPScore (caption grounding, orig image):       {ground_scores.mean():.4f}")
print(f"Best / worst cycle sim: {cycle_sims.max():.3f} / {cycle_sims.min():.3f}")

In [ ]:
# ── Visual inspection: original vs reconstruction, with scores ────────────
import matplotlib.pyplot as plt
import numpy as np

order = np.argsort(-cycle_sims)               # best matches first
show  = list(order[:3]) + list(order[-2:])    # 3 best + 2 worst
fig, axes = plt.subplots(len(show), 2, figsize=(8, 4 * len(show)))
for row, idx in enumerate(show):
    axes[row, 0].imshow(originals[idx]);  axes[row, 0].axis("off")
    axes[row, 0].set_title(f"ORIGINAL #{idx}", fontsize=10)
    axes[row, 1].imshow(recons[idx]);     axes[row, 1].axis("off")
    axes[row, 1].set_title(f"RECONSTRUCTED  cyc={cycle_sims[idx]:.3f}", fontsize=10)
    print(f"#{idx} cyc={cycle_sims[idx]:.3f} ground={ground_scores[idx]:.3f}")
    print("   CAP:", captions[idx][:160])
plt.tight_layout(); plt.show()

## Reading the results

- **CLIP image–image cosine** is the cycle-reconstruction fidelity: did the caption
  carry enough distinguishing detail to regenerate a semantically similar figure?
  Compare this number across checkpoints (e.g. SFT-only vs GRPO) — a higher mean
  means the GRPO objective pushed captions toward more image-grounded specifics.
- **CLIPScore(caption, original)** is the direct grounding metric from your training
  reward, on held-out data — a sanity check that captions still align with the image.

**Caveats / knobs:**
- General text-to-image models don't render real plots/axes faithfully, so absolute
  cosine values are modest — use them comparatively. For sharper signal, swap
  `sd-turbo` for `sdxl-turbo`, raise `num_inference_steps`, or compare against the
  SFT-only adapter to isolate GRPO's contribution.
- To eval on more samples, raise `N_EVAL` in the dataset cell.
- If you have a real `scicap_val.csv`, use the optional block in the dataset cell.